# Pemodelan — XGBoost Kuantil

## 1. Konfigurasi

In [ ]:
# Notebook ini menjalankan XGBoost kuantil dari awal sampai akhir: benchmark (§4), pencarian
# hyperparameter (§5), walk-forward lima fold (§6), model final (§7), ringkasan hasil (§8), dan
# perbandingan langsung dengan Random Forest (§9). Desember 2025 terkunci sebagai test set dan
# tidak dinilai di sini.
#
# Kode modelnya ada di §2 notebook ini, salinan verbatim dari utils/modelling/model_xgboost.py;
# §10 membandingkan keduanya dan menyebut fungsi mana yang menyimpang kalau ada. Mesin bersama
# yang dipakai ketiga notebook modeling tetap diimpor dari utils — modeling_prep, walk_forward,
# evaluation, model_common, purging, run_config — karena menyalin 2.129 baris itu ke tiga
# notebook lebih mahal daripada pemisahan yang dibelinya (keputusan pemilik proyek 2026-08-26).
#
# Desain: docs/superpowers/specs/2026-08-19-xgboost-modeling-design.md
# Rencana: docs/superpowers/plans/2026-08-19-xgboost-modeling.md
# Hasil terukur: belum ada untuk kriteria multi-kuantil. Angka era kuantil-tunggal
# diarsipkan di docs/bak/hasil-modeling-xgb.single-quantile.bak.md dan TIDAK sebanding
# dengan K1 — dokumen Fase 3 ditulis setelah pencarian di §5 selesai.
import sys
from functools import partial
from pathlib import Path
from typing import Callable, Iterable, Optional

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from xgboost.core import XGBoostError


def find_base_dir(start=None) -> Path:
    """Cari root repo — folder pertama ke atas yang berisi `dataset/csv/`."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "csv").is_dir():
            return candidate
    raise RuntimeError(f"Root repo tidak ditemukan dari {start}")


BASE_DIR = find_base_dir()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from utils.modelling import evaluation, model_common, modeling_prep, purging, run_config
from utils.modelling import walk_forward

print(f"BASE_DIR = {BASE_DIR}")

## 2. Definisi model XGBoost kuantil

In [ ]:
# XGBoost pada service level 0,9.
#
# `reg:quantileerror` mengoptimalkan pinball loss yang sama dengan kriteria pemilihan model,
# jadi objective pelatihan dan kriteria seleksi sejalan — hal yang tidak berlaku untuk model
# squared-error yang baru dimintai kuantil tinggi belakangan.
#
# Yang membuat wrapper ini lebih dari sekadar pemanggilan tipis adalah jumlah rondenya. Boosting
# overfit kalau berjalan terlalu lama, sehingga jumlah ronde itu sendiri adalah keputusan
# regularisasi — dan tempat yang paling jelas untuk mengambilnya, yaitu fold validasi, justru
# tempat yang akan bocor. Karena itu early stopping berjalan pada ekor training window yang
# ditahan, lalu model difit ulang pada seluruh baris training di jumlah ronde yang dipilih ekor
# tadi. Dua fit per fold, supaya XGBoost akhirnya dilatih pada populasi yang sama dengan yang
# dilihat Random Forest dan perbandingannya tetap setara.
QUANTILE = 0.9


QUANTILES = evaluation.QUANTILE_SET_A


ES_TAIL_DAYS = 30


EARLY_STOPPING_ROUNDS = 50


MAX_ROUNDS = 2000


DEFAULT_PARAMS = {
    "max_depth": 6,
    "learning_rate": 0.05,
    "min_child_weight": 10,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_lambda": 1.0,
    "encoding": "ordinal",
    "log_target": False,
    "random_state": 42,
}


SEARCH_SPACE = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 10, 50],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.5, 0.7, 1.0],
    "reg_lambda": [1.0, 10.0],
    "encoding": ["ordinal", "native", "one_hot"],
    "log_target": [False, True],
}


ESTIMATOR_KEYS = ("max_depth", "learning_rate", "min_child_weight",
                  "subsample", "colsample_bytree", "reg_lambda", "random_state")


DEFAULT_DEVICE = "cpu"


split_early_stopping = model_common.split_early_stopping


ENCODINGS = ("ordinal", "native", "one_hot")

In [ ]:
def _present(frame: pd.DataFrame, idx_cols: Optional[list]) -> list:
    idx_cols = model_common.IDX_COLS if idx_cols is None else idx_cols
    return [col for col in idx_cols if col in frame.columns]


def training_categories(
    train_X: pd.DataFrame,
    idx_cols: Optional[list] = None,
) -> dict:
    """The category levels each encoded column had in training, sorted.

    Recorded in the bundle so a reloaded model rebuilds the identical dtype.
    Category codes are positional: rebuilding a column from a different level
    list silently renumbers every row.
    """
    return {
        col: sorted(train_X[col].dropna().unique().tolist())
        for col in _present(train_X, idx_cols)
    }


def _as_categorical(frame: pd.DataFrame, categories: dict) -> pd.DataFrame:
    out = frame.copy()
    for col, levels in categories.items():
        if col in out.columns:
            out[col] = out[col].astype(pd.CategoricalDtype(categories=levels))
    return out


def encode(
    train_X: pd.DataFrame,
    valid_X: pd.DataFrame,
    encoding: str,
    idx_cols: Optional[list] = None,
) -> tuple:
    """Prepare both matrices under one of the three searched encodings.

    Returns `(train_out, valid_out, enable_categorical)`. In every mode the
    validation columns are forced onto the training columns: a category
    present only in validation would otherwise shift every column after it,
    and the booster would read the wrong feature at each position — silently,
    since the shapes still line up.

    Under `native`, a level unseen in training becomes NaN rather than
    borrowing another level's code. XGBoost consumes NaN natively, so an
    unknown category is treated as missing, which is what it is.
    """
    if encoding == "ordinal":
        return train_X, valid_X.reindex(columns=train_X.columns), False
    if encoding == "native":
        categories = training_categories(train_X, idx_cols)
        train_out = _as_categorical(train_X, categories)
        valid_out = _as_categorical(valid_X.reindex(columns=train_X.columns),
                                    categories)
        return train_out, valid_out, True
    if encoding == "one_hot":
        train_out, valid_out = model_common.expand_one_hot(
            train_X, valid_X, idx_cols=_present(train_X, idx_cols)
        )
        return train_out, valid_out, False
    raise ValueError(f"encoding tidak dikenal: {encoding!r}, pilih dari {ENCODINGS}")


def apply_encoding(
    X: pd.DataFrame,
    encoding: str,
    columns: list,
    categories: dict,
    idx_cols: Optional[list] = None,
) -> tuple:
    """Encode a frame for prediction against a layout recorded at fit time.

    A booster reloaded next month against columns in a different order does
    not fail — it predicts confidently from the wrong features, which is
    worse. `columns` is the authority here, not whatever the caller passed in.
    """
    if encoding == "one_hot":
        out = pd.get_dummies(X, columns=_present(X, idx_cols))
    elif encoding == "native":
        out = _as_categorical(X, categories)
    elif encoding == "ordinal":
        out = X
    else:
        raise ValueError(f"encoding tidak dikenal: {encoding!r}, pilih dari {ENCODINGS}")

    out = out.reindex(columns=columns, fill_value=0)
    if encoding == "native":
        out = _as_categorical(out, categories)
    return out, encoding == "native"

In [ ]:
def build_estimator(
    params: dict,
    n_estimators: int,
    enable_categorical: bool = False,
    early_stopping_rounds: Optional[int] = None,
    quantiles: tuple = QUANTILES,
    device: str = DEFAULT_DEVICE,
) -> XGBRegressor:
    """A regressor whose training objective is the metric it is judged on.

    `quantile_alpha` takes the whole grid rather than one point. Every
    quantile is then fitted inside one booster against the same loss K1
    averages, so training objective and selection criterion stay the same
    function — the property a squared-error model asked for a high quantile
    afterwards does not have, and the one that would be lost by fitting
    nineteen separate single-quantile boosters.
    """
    kwargs = {key: params[key] for key in ESTIMATOR_KEYS if key in params}
    return XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=np.asarray(quantiles, dtype=float),
        tree_method="hist",
        device=device,
        n_estimators=n_estimators,
        enable_categorical=enable_categorical,
        early_stopping_rounds=early_stopping_rounds,
        n_jobs=-1,
        **kwargs,
    )


def _target(frame: pd.DataFrame, params: dict) -> np.ndarray:
    return model_common.train_target(frame, log_target=params["log_target"])


def _as_matrix(prediction, n_rows: int, n_quantiles: int) -> np.ndarray:
    """XGBoost drops the second axis on a one-point grid; this puts it back.

    Tahap B can hand out a grid of one (evaluation.quantile_set_b() dedupes),
    and every consumer downstream indexes columns. A silently 1-D return there
    would fail far from its cause.
    """
    return np.asarray(prediction, dtype=float).reshape(n_rows, n_quantiles)


def make_fit_predict(
    params: Optional[dict] = None,
    feature_cols: Optional[list] = None,
    quantiles: tuple = QUANTILES,
    tail_days: int = ES_TAIL_DAYS,
    early_stopping_rounds: int = EARLY_STOPPING_ROUNDS,
    max_rounds: int = MAX_ROUNDS,
    idx_cols: Optional[list] = None,
    device: str = DEFAULT_DEVICE,
) -> Callable[[pd.DataFrame, pd.DataFrame], np.ndarray]:
    """The callable walk_forward.run_fold() injects.

    Two fits. The first runs on the purged fit rows with the tail as its eval
    set and reports where early stopping landed. The second discards that
    booster and refits on every training row at exactly that round count, so
    the model that produces the reported predictions has seen the same
    population the Random Forest was trained on.

    Under `log_target`, the early-stopping metric is computed on the log
    scale. That is sound: early stopping only chooses a round count *within*
    one candidate. Candidates are compared to each other by pinball on the
    original scale, after inversion.

    Round counts are recorded on the returned callable rather than returned,
    because `walk_forward` accepts predictions and nothing else — and the
    spread of round counts across folds is worth reporting.

    Early stopping now watches the mean quantile loss across the whole grid
    rather than the loss at 0.9. One round count still serves every point,
    which is a real constraint: a booster cannot stop at round 300 for τ=0.05
    and round 700 for τ=0.95. The alternative — nineteen independently stopped
    boosters — buys per-point round counts at the price of nineteen fits and a
    guaranteed loss of the shared structure, and this project pays for the
    shared structure.
    """
    params = {**DEFAULT_PARAMS, **(params or {})}
    feature_cols = feature_cols or modeling_prep.FEATURE_COLS
    quantiles = tuple(quantiles)

    def fit_predict(train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
        model_common.assert_no_nan(train, feature_cols)
        model_common.assert_no_nan(valid, feature_cols)

        fit_rows, es_rows = split_early_stopping(train, tail_days=tail_days)
        fit_X, es_X, enable = encode(fit_rows[feature_cols], es_rows[feature_cols],
                                     params["encoding"], idx_cols=idx_cols)
        probe = build_estimator(params, max_rounds, enable_categorical=enable,
                                early_stopping_rounds=early_stopping_rounds,
                                quantiles=quantiles, device=device)
        probe.fit(fit_X, _target(fit_rows, params),
                  eval_set=[(es_X, _target(es_rows, params))], verbose=False)
        best_iteration = int(probe.best_iteration) + 1
        fit_predict.best_iterations.append(best_iteration)

        train_X, valid_X, enable = encode(train[feature_cols], valid[feature_cols],
                                          params["encoding"], idx_cols=idx_cols)
        model = build_estimator(params, best_iteration, enable_categorical=enable,
                                quantiles=quantiles, device=device)
        model.fit(train_X, _target(train, params), verbose=False)

        prediction = _as_matrix(model.predict(valid_X), len(valid_X),
                                len(quantiles))
        if params["log_target"]:
            prediction = modeling_prep.inverse_log_target(prediction)
        # A negative shipment quantity is not a thing. No sort: crossing is
        # measured by evaluation.crossing_rate(), and sorting here would drive
        # that measurement to zero without fixing anything.
        return np.clip(prediction, 0.0, None)

    fit_predict.best_iterations = []
    return fit_predict

In [ ]:
MODEL_FILE = str(BASE_DIR / "models/xgboost_q90.joblib")


BEST_PARAMS_FILE = str(BASE_DIR / "dataset/model_ready/xgb_best_params.json")


def fit_final(
    df: pd.DataFrame,
    params: dict,
    feature_cols: Optional[list] = None,
    quantiles: tuple = QUANTILES,
    tail_days: int = ES_TAIL_DAYS,
    early_stopping_rounds: int = EARLY_STOPPING_ROUNDS,
    max_rounds: int = MAX_ROUNDS,
    idx_cols: Optional[list] = None,
    device: str = DEFAULT_DEVICE,
    date_col: str = modeling_prep.DATE_COL,
    test_start: pd.Timestamp = modeling_prep.TEST_START,
) -> dict:
    """Fit on every eligible row before December, purged at that boundary.

    Eligibility comes from `walk_forward.eligible_rows`, not from a date filter
    written here. The rows this model is finally trained on have to be the rows
    it was scored on, and the scoring cuts are not just the date: the first 28
    days of each segment have no usable lag window, and the last few days have
    no target at all, because the lead-time sum runs past the end of the data.

    Same two-fit protocol as walk-forward. The bundle records the training
    column order, the encoding, and the category levels, because a booster
    reloaded next month against a different layout does not fail — it predicts
    confidently from the wrong features, which is worse.
    """
    params = {**DEFAULT_PARAMS, **params}
    feature_cols = feature_cols or modeling_prep.FEATURE_COLS
    quantiles = tuple(quantiles)

    frame = walk_forward.eligible_rows(df, date_col=date_col, test_start=test_start)
    frame = frame[purging.lookahead_safe_mask(frame, test_start, date_col=date_col)]
    model_common.assert_no_nan(frame, feature_cols)

    fit_rows, es_rows = split_early_stopping(frame, tail_days=tail_days,
                                             date_col=date_col)
    fit_X, es_X, enable = encode(fit_rows[feature_cols], es_rows[feature_cols],
                                 params["encoding"], idx_cols=idx_cols)
    probe = build_estimator(params, max_rounds, enable_categorical=enable,
                            early_stopping_rounds=early_stopping_rounds,
                            quantiles=quantiles, device=device)
    probe.fit(fit_X, _target(fit_rows, params),
              eval_set=[(es_X, _target(es_rows, params))], verbose=False)
    best_iteration = int(probe.best_iteration) + 1

    train_X, _, enable = encode(frame[feature_cols], frame[feature_cols],
                                params["encoding"], idx_cols=idx_cols)
    model = build_estimator(params, best_iteration, enable_categorical=enable,
                            quantiles=quantiles, device=device)
    model.fit(train_X, _target(frame, params), verbose=False)

    return {
        "model": model,
        "params": params,
        "feature_cols": feature_cols,
        "columns": list(train_X.columns),
        "categories": training_categories(frame[feature_cols], idx_cols=idx_cols),
        "idx_cols": idx_cols,
        "encoding": params["encoding"],
        "log_target": params["log_target"],
        "best_iteration": best_iteration,
        "quantiles": quantiles,
        # Provenance, not configuration. A bundle says which device produced
        # it so a wall clock read months later is attributable, and so a
        # winner chosen on one device is never silently refitted on another.
        "device": device,
        **model_common.target_provenance(),
        "n_train": int(len(frame)),
    }


def predict_bundle(bundle: dict, frame: pd.DataFrame) -> np.ndarray:
    """Predict with a fitted bundle, forcing the recorded column order.

    The grid comes from the bundle rather than this module's constant: the
    booster's output columns are fixed at fit time, and reading them against a
    grid that has since moved would relabel every column silently.
    """
    features, _ = apply_encoding(frame[bundle["feature_cols"]],
                                 bundle["encoding"], bundle["columns"],
                                 bundle["categories"],
                                 idx_cols=bundle["idx_cols"])
    prediction = _as_matrix(bundle["model"].predict(features), len(features),
                            len(bundle["quantiles"]))
    if bundle["log_target"]:
        prediction = modeling_prep.inverse_log_target(prediction)
    return np.clip(prediction, 0.0, None)


def save_bundle(bundle: dict, path: str = MODEL_FILE) -> None:
    model_common.save_bundle(bundle, path)


def load_bundle(path: str = MODEL_FILE) -> dict:
    return model_common.load_bundle(path)


def save_best_params(params: dict, path: str = BEST_PARAMS_FILE) -> None:
    model_common.save_best_params(params, path)

In [ ]:
SEARCH_FOLDS = (3, 5)


N_CANDIDATES = 30


select_best = model_common.select_best


def sample_search_space(
    n_candidates: int = N_CANDIDATES,
    seed: int = 42,
    space: Optional[dict] = None,
) -> list:
    """Distinct parameter sets drawn at random from SEARCH_SPACE.

    No affordability screen: `hist` holds a quantized feature matrix — tens of
    megabytes at this size — so there is no analogue of the quantile forest's
    leaf-storage bound to screen against.
    """
    return model_common.sample_search_space(
        space=SEARCH_SPACE if space is None else space,
        defaults=DEFAULT_PARAMS,
        n_candidates=n_candidates,
        seed=seed,
        screen=None,
    )


def run_search(
    df: pd.DataFrame,
    candidates: list,
    folds: tuple = SEARCH_FOLDS,
    quantiles: tuple = QUANTILES,
    model_name: str = "xgboost",
    feature_cols: Optional[list] = None,
    verbose: bool = True,
    checkpoint_path: Optional[str] = None,
    resume: bool = True,
    only: Optional[Iterable[int]] = None,
    provenance: Optional[dict] = None,
    device: str = DEFAULT_DEVICE,
) -> pd.DataFrame:
    """Score every XGBoost candidate on the search folds.

    XGBoostError joins the caught types: a candidate whose parameter
    combination the library rejects should be recorded and skipped, exactly
    like an over-budget forest, rather than ending a multi-hour run.
    """
    return model_common.run_search(
        df, candidates,
        # `model_common.run_search` calls the factory with three keywords and
        # no slot for a device, exactly as the LSTM has no slot for its panel.
        # Binding it here keeps that signature the same for all three models.
        make_fit_predict=partial(make_fit_predict, device=device),
        search_space=SEARCH_SPACE, folds=folds, quantiles=quantiles,
        model_name=model_name, feature_cols=feature_cols, verbose=verbose,
        checkpoint_path=checkpoint_path, resume=resume,
        only=only, provenance=provenance,
        catch=(MemoryError, ValueError, XGBoostError),
    )

## 3. Setelan run & data model-ready

In [ ]:
# Di mana boosting dijalankan. "cpu" adalah default dan satu-satunya
# angka yang pernah diukur sejauh ini; "cuda" untuk mesin sewaan.
# Grid 19 titik membangun 19 pohon per ronde boosting (T-14), jadi
# tahap ini yang paling mahal di Fase 3.
#
# SATU MODEL = SATU DEVICE, dari benchmark sampai fit final. Kandidat
# yang dipilih di satu device lalu difit ulang di device lain tidak
# dipilih di bawah aritmetika yang menghasilkannya — hist di GPU dan
# hist di CPU tidak membangun pohon yang identik. Device tercatat di
# bundle supaya itu terbaca, bukan diasumsikan.
#
# Keempat setelan di bawah dibaca dari environment; tanpa satu pun di
# antaranya, notebook ini berperilaku persis seperti sebelum jalur cloud
# ada — termasuk nama berkas yang ditulisnya. Untuk memecah pencarian:
#
#   FORECAST_SHARD=0-14 FORECAST_DEVICE=cuda:0 \
#   FORECAST_MODEL_INPUT=/kaggle/input/forecast-scm/model_input.parquet \
#   FORECAST_CHECKPOINT_DIR=/kaggle/working jupyter nbconvert ...
#
# Rencana mesin per tahap ada di
# docs/superpowers/specs/2026-08-24-distributed-gpu-training-design.md.
DEVICE = run_config.device(DEFAULT_DEVICE)
SHARD = run_config.shard()
SEARCH_FILE = run_config.search_checkpoint("xgb")
RESULTS_FILE = run_config.checkpoint_path("xgb_walk_forward_results.csv")

df = pd.read_parquet(run_config.model_input_path())
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(run_config.describe(DEVICE))

# Path artefak datang dari run_config, bukan dari konstanta modul, supaya FORECAST_CHECKPOINT_DIR
# berlaku untuk seluruh sel — termasuk §8 dan §9 yang membaca ulang tabel hasil. Tanpa env var
# apa pun keduanya menunjuk berkas yang sama.
print(f"SEARCH_FILE  = {SEARCH_FILE}")
print(f"RESULTS_FILE = {RESULTS_FILE}")

## 4. Benchmark

In [ ]:
# Satu putaran dua-fit di fold 5 dengan DEFAULT_PARAMS, untuk mengukur ongkos sebelum 60 fit
# pencarian dijalankan dan melihat di ronde berapa early stopping mendarat. Angkanya dicatat
# di dokumen hasil XGBoost Fase 3 yang ditulis setelah pencarian selesai.
import resource
import time

split = walk_forward.prepare_fold(df, 5)
train, valid = split["train"], split["valid"]
fit_rows, es_rows = split_early_stopping(train)
print(f"train {len(train):,} rows -> fit {len(fit_rows):,} + tail {len(es_rows):,}")
print(f"valid {len(valid):,} rows")
print(f"QUANTILE_SET: {len(QUANTILES)} titik, {QUANTILES[0]}..{QUANTILES[-1]}")

fit_predict = make_fit_predict(dict(DEFAULT_PARAMS), device=DEVICE)
start = time.time()
prediction = fit_predict(train, valid)
elapsed = time.time() - start

peak_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss  # bytes on macOS
headline = min(range(len(QUANTILES)),
               key=lambda i: abs(QUANTILES[i] - evaluation.DEFAULT_ALPHA))
print(f"best_iteration {fit_predict.best_iterations[0]} of {MAX_ROUNDS}")
print(f"wall time {elapsed / 60:.1f} min (both fits)")
print(f"peak RSS  {peak_bytes / 1024 ** 3:.2f} GB")
print(f"prediction shape {prediction.shape} (baris x titik kuantil)")
print(f"di tau=0.9: mean {prediction[:, headline].mean():.2f}, "
      f"max {prediction[:, headline].max():.2f}")
# Head komposit tidak punya jaminan monotonisitas struktural: angka ini
# diukur, bukan diasumsikan nol, dan di atas beberapa persen ia sinyal bahwa
# arctan pinball loss (Sluijterman dkk. 2024) sepadan dengan kerumitannya.
print(f"crossing_rate {evaluation.crossing_rate(prediction, QUANTILES):.4f}")

## 5. Pencarian hyperparameter

In [ ]:
# 30 kandidat dari ruang 2.592 kombinasi, dinilai di fold 3 dan 5 dengan pinball@0,9 gabungan.
# Hasil yang dilaporkan datang dari walk-forward lima fold di §6, bukan dari sini — menilai di
# fold yang memilih pemenang akan optimistis.
# Ke-30 kandidat dijalankan ulang: objective-nya kini quantile_alpha =
# seluruh QUANTILE_SET, jadi skor lama (pinball@0,9) bukan K1 dan tidak
# sebanding. Artefak run kuantil-tunggal sudah diganti nama menjadi
# `*.single-quantile.bak.*` (2026-08-24) sesudah guard checkpoint
# diverifikasi berbunyi, jadi sel ini mulai dari nol. Kalau berkas tanpa
# kolom `headline_quantile` muncul lagi di jalur checkpoint, guard yang
# sama akan menolaknya dalam hitungan detik — itu perilaku yang diinginkan.
#
# Dengan FORECAST_SHARD diset, mesin ini hanya menjalankan kandidat yang
# jadi bagiannya, tetapi penomorannya tetap absolut terhadap seed 42 —
# itulah yang membuat dua shard bisa disatukan `model_common.merge_shards()`
# nanti. Kolom `device` dan `commit` ikut ditulis ke tiap baris supaya
# angkanya dapat ditelusuri ke mesin yang melahirkannya.
candidates = sample_search_space(N_CANDIDATES, seed=42)
search_results = run_search(df, candidates, folds=SEARCH_FOLDS,
                                checkpoint_path=SEARCH_FILE, device=DEVICE,
                                only=SHARD,
                                provenance=run_config.provenance(DEVICE))
search_results.to_csv(SEARCH_FILE, index=False)
# `pinball` di sini adalah K1. Kolom *_headline dibaca di tau=0,9.
search_results.sort_values("pinball").head(10)

## 6. Walk-forward final

In [ ]:
# Konfigurasi pemenang di kelima fold, melawan ketiga baseline naive pada baris yang identik.
# Berhenti di sini kalau ini run bershard: `select_best()` di bawah akan
# memilih pemenang dari sebagian kandidat saja, dan hasilnya akan tampak
# sepenuhnya wajar. Gabungkan dulu seluruh shard di satu mesin —
#
#   from utils.modelling import model_common
#   merged = model_common.merge_shards(
#       ["xgb_search_results.shard-0-14.csv",
#        "xgb_search_results.shard-15-29.csv"],
#       candidates, SEARCH_SPACE)
#
# — lalu jalankan sel ini di mesin itu, tanpa FORECAST_SHARD.
#
# Walk-forward dan fit final ketiga model dijalankan di **satu mesin yang
# sama** (Mac lokal), karena wall time-nya masuk K3 dan K3-lah yang
# menentukan pemenang saat K1 seri — lihat Bagian 1 spec eksekusi
# terdistribusi. Ia juga tidak punya checkpoint: sesi yang terpotong di
# tengah kehilangan seluruhnya.
assert SHARD is None, (
    "run bershard: jangan pilih pemenang dari sebagian kandidat — "
    "gabungkan seluruh shard dengan model_common.merge_shards() lebih dulu"
)

best = select_best(search_results, candidates)
save_best_params(best)
print(best)

fit_predict = make_fit_predict(best, device=DEVICE)
results = walk_forward.run_walk_forward(df, fit_predict, model_name="xgboost",
                                        quantiles=QUANTILES)
results.to_csv(RESULTS_FILE, index=False)

print("best_iteration per fold:", fit_predict.best_iterations)
print(f"K1 (rata-rata pinball lintas {len(QUANTILES)} kuantil): "
      f"{walk_forward.pooled_k1(results, 'xgboost'):.4f}")

overall = results[results["group_col"].isna()]
(overall[(overall["quantile"] - evaluation.DEFAULT_ALPHA).abs() < 1e-9]
 .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

## 7. Model final

In [ ]:
bundle = fit_final(df, best, device=DEVICE)
save_bundle(bundle)
print(f"trained on {bundle['n_train']:,} rows, "
      f"{len(bundle['columns'])} columns, encoding {bundle['encoding']}, "
      f"{bundle['best_iteration']} rounds, device {bundle['device']}, "
      f"{len(bundle['quantiles'])} titik kuantil "
      f"({bundle['quantiles'][0]}..{bundle['quantiles'][-1]})")

## 8. Hasil

In [ ]:
# Tiga potongan, masing-masing melawan ketiga baseline naive pada baris identik. Satu angka
# global menyesatkan di data yang 44% targetnya nol.
results = pd.read_csv(RESULTS_FILE)
HEADLINE = evaluation.DEFAULT_ALPHA

print("=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} K1 {walk_forward.pooled_k1(results, model):7.4f}")

print("\n=== per fold, K1 ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

print(f"\n=== per fold, pinball di tau={HEADLINE} (angka headline B-9) ===")
headline_rows = results[results["group_col"].isna()
                        & ((results["quantile"] - HEADLINE).abs() < 1e-9)]
print(headline_rows.pivot_table(index="model", columns="fold_id",
                                values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (K1, pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    # Kolom dipilih sebelum apply: tanpa itu pandas ikut menyertakan kolom
    # pengelompokan dan mengeluarkan FutureWarning di setiap sel.
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)[["weighted", "n"]]
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

# pooled_metric menolak dirata-ratakan lintas kuantil untuk metrik selain
# pinball/crossing_rate — coverage di 0,05 dan di 0,95 menjawab pertanyaan
# yang berbeda. Jadi ketiganya dibaca di tau headline, eksplisit.
print(f"\n=== coverage / fill rate di tau={HEADLINE} (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage', quantile=HEADLINE):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate', quantile=HEADLINE):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units', quantile=HEADLINE):9.1f}  "
          f"crossing {walk_forward.pooled_metric(results, model, 'crossing_rate'):6.4f}")

print("\n=== K2: coverage per titik kuantil (xgboost) ===")
print(walk_forward.coverage_by_quantile(results, "xgboost").round(4)
      .to_string(index=False))

## 9. Head-to-head lawan Random Forest

In [ ]:
# Sah dilakukan karena kedua model dinilai di baris yang identik — dijamin oleh
# walk_forward.eligible_rows(), bukan oleh disiplin. Fold 1, 2, 4 adalah potongan yang bersih:
# keduanya memilih pemenang di fold 3 dan 5.
rf_results = pd.read_csv(rf.RESULTS_FILE)
combined = pd.concat([results, rf_results], ignore_index=True)
HEADLINE = evaluation.DEFAULT_ALPHA

for label, folds in (("semua fold", None), ("fold 1/2/4 (bersih)", (1, 2, 4))):
    print(f"=== {label} ===")
    for model in ("xgboost", "random_forest", "naive_roll_mean_7"):
        rows = combined[(combined["model"] == model) & combined["group_col"].isna()]
        if rows.empty:
            continue
        print(f"{model:20s} "
              f"K1 {walk_forward.pooled_k1(combined, model, folds):6.3f}  "
              f"mae@0.9 {walk_forward.pooled_metric(combined, model, 'mae', folds, quantile=HEADLINE):7.3f}  "
              f"coverage@0.9 {walk_forward.pooled_metric(combined, model, 'coverage', folds, quantile=HEADLINE):6.3f}")
    print()

# Lantai naif dinilai di seluruh 19 titik juga — ia ramalan titik, jadi ia
# membayar penalti pinball karena berpura-pura satu angkanya adalah setiap
# kuantil sekaligus. K1 baseline karenanya BUKAN angka 6,56 pinball@0,9 yang
# tercatat di dokumen hasil lama; keduanya tidak boleh disandingkan (T-10).

## 10. *(Opsional)* Cek sinkron dengan `utils/`

In [ ]:
import inspect

from utils.modelling import model_xgboost as _ref

# SEARCH_FILE dan RESULTS_FILE sengaja datang dari run_config di §3, jadi nilainya boleh berbeda
# dari konstanta modul begitu FORECAST_CHECKPOINT_DIR diset. Sisanya harus identik.
_LEWATI = {"find_base_dir", "SEARCH_FILE", "RESULTS_FILE",
           "_LEWATI", "_ref", "_beda", "_n_fungsi", "_n_konstanta"}

_beda, _n_fungsi, _n_konstanta = [], 0, 0
for nama, obj in sorted(globals().items()):
    if nama in _LEWATI or nama.startswith("__") or not hasattr(_ref, nama):
        continue
    ref = getattr(_ref, nama)
    if inspect.isfunction(obj):
        _n_fungsi += 1
        if inspect.getsource(obj) != inspect.getsource(ref):
            _beda.append(nama)
    elif nama.isupper():
        _n_konstanta += 1
        if obj != ref:
            _beda.append(nama)

if _beda:
    print("BERBEDA dari utils/ — salin ulang atau samakan: " + ", ".join(_beda))
else:
    print(f"Sinkron: {_n_fungsi} fungsi + {_n_konstanta} konstanta identik dengan "
          f"utils/modelling/model_xgboost.py")